# Supervisor revision checks

This notebook reproduces the focused empirical checks requested in the final supervisor review. It uses the saved Chapter 5 outputs and **does not refit any forecasting model**.

It produces:

- paired confidence intervals for point-forecast accuracy differences;
- ARIMA/SARIMA non-convergence sensitivity summaries;
- TiRex-2 calibration intervals and quantile-crossing diagnostics;
- terminal on-hand and pipeline-inventory summaries;
- product- and store-cluster bootstrap checks; and
- the baseline cost-fill-rate/Pareto-frontier figure.

Before running the notebook, extract both supporting-output archives as described in the package `README.md`. Run all cells from top to bottom.


## 1. Software environment


In [ ]:
import platform
import sys
from importlib.metadata import PackageNotFoundError, version

print("Python:", sys.version.replace("\n", " "))
print("Platform:", platform.platform())
for package_name in ["numpy", "pandas", "matplotlib"]:
    try:
        print(f"{package_name}: {version(package_name)}")
    except PackageNotFoundError:
        print(f"{package_name}: not installed")


## 2. Imports and fixed settings

The bootstrap uses 2,000 resamples and seed 2026, matching the thesis and the command-line script.


In [ ]:
from pathlib import Path
import itertools
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

MODEL_ORDER = [
    "seasonal_naive",
    "ets",
    "arima_sarima",
    "xgboost",
    "tirex2",
]

MODEL_LABELS = {
    "seasonal_naive": "Seasonal naïve",
    "ets": "ETS",
    "arima_sarima": "ARIMA/SARIMA",
    "xgboost": "XGBoost",
    "tirex2": "TiRex-2",
}

CLASS_ORDER = ["Smooth", "Erratic", "Intermittent", "Lumpy"]
QUANTILE_LEVELS = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]
BOOTSTRAP_RESAMPLES = 2_000
RANDOM_SEED = 2026


## 3. Paths and input checks

The notebook searches upward from the current working directory for the package root. The default input directory is `data/chapter5_results`, created by extracting the two supplied result archives. Optional environment variables can override the defaults:

- `THESIS_CORE_DIR`
- `THESIS_QUANTILE_DIR`
- `THESIS_SAMPLE_FILE`
- `THESIS_REVISION_OUTPUT_DIR`
- `THESIS_FIGURE_DIR`


In [ ]:
def locate_package_root() -> Path:
    """Locate the reproducibility-package root from a notebook or package directory."""

    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (
            (candidate / "README.md").is_file()
            and (candidate / "data" / "selected_sample_ids.csv").is_file()
        ):
            return candidate
    raise FileNotFoundError(
        "Could not locate the reproducibility-package root. "
        "Start Jupyter inside the extracted package or set the paths below manually."
    )


PACKAGE_ROOT = locate_package_root()
CORE_DIR = Path(
    os.environ.get("THESIS_CORE_DIR", PACKAGE_ROOT / "data" / "chapter5_results")
).expanduser().resolve()
QUANTILE_DIR = Path(
    os.environ.get("THESIS_QUANTILE_DIR", CORE_DIR)
).expanduser().resolve()
SAMPLE_FILE = Path(
    os.environ.get(
        "THESIS_SAMPLE_FILE",
        PACKAGE_ROOT / "data" / "selected_sample_ids.csv",
    )
).expanduser().resolve()
OUTPUT_DIR = Path(
    os.environ.get(
        "THESIS_REVISION_OUTPUT_DIR",
        PACKAGE_ROOT / "outputs" / "revision_checks",
    )
).expanduser().resolve()
FIGURE_DIR = Path(
    os.environ.get("THESIS_FIGURE_DIR", PACKAGE_ROOT / "figures")
).expanduser().resolve()

required_files = [
    SAMPLE_FILE,
    CORE_DIR / "point_forecast_series_origin_metrics.csv",
    CORE_DIR / "arima_sarima_fit_diagnostics.csv",
    CORE_DIR / "inventory_weekly_arima_sarima_additional_diagnostics.csv",
    CORE_DIR / "inventory_series_cost_scenarios.csv",
    CORE_DIR / "inventory_summary.csv",
    CORE_DIR / "inventory_policy_timing_series_results.csv",
    QUANTILE_DIR / "tirex2_daily_quantile_evaluation_rows.csv",
    QUANTILE_DIR / "residual_based_protection_quantiles.csv",
]
missing_files = [path for path in required_files if not path.is_file()]
if missing_files:
    missing_text = "\n".join(f"- {path}" for path in missing_files)
    raise FileNotFoundError(
        "Required inputs are missing. Extract both supporting-output archives first:\n"
        + missing_text
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

sample = pd.read_csv(SAMPLE_FILE)
sample["id"] = sample["id"].astype(str)
sample = sample.rename(columns={"demand_pattern": "demand_class"})

print("Package root:", PACKAGE_ROOT)
print("Core results:", CORE_DIR)
print("Quantile results:", QUANTILE_DIR)
print("Selected series:", sample["id"].nunique())


## 4. Paired uncertainty intervals for point-forecast accuracy

SKU-store series are resampled as intact temporal blocks, stratified by demand class. The same resampled series are used for both models in each comparison, preserving the paired design.


In [ ]:
def percentile_interval(values: np.ndarray) -> tuple[float, float]:
    """Return the two-sided 95% percentile interval."""

    lower, upper = np.quantile(values, [0.025, 0.975])
    return float(lower), float(upper)


def normalise_model_names(values: pd.Series) -> pd.Series:
    """Normalise the model labels used across saved output files."""

    return (
        values.astype(str)
        .str.lower()
        .str.replace("-", "_", regex=False)
        .str.replace("/", "_", regex=False)
        .str.replace(" ", "_", regex=False)
        .replace({"tirex_2": "tirex2", "seasonal_naïve": "seasonal_naive"})
    )


def stratified_bootstrap_ids(
    sample: pd.DataFrame,
    rng: np.random.Generator,
) -> list[np.ndarray]:
    """Create shared series bootstrap indices, stratified by demand class."""

    class_ids = {
        demand_class: sample.loc[
            sample["demand_class"].eq(demand_class), "id"
        ].to_numpy()
        for demand_class in CLASS_ORDER
    }

    draws: list[np.ndarray] = []
    for _ in range(BOOTSTRAP_RESAMPLES):
        draw = np.concatenate(
            [
                rng.choice(ids, size=len(ids), replace=True)
                for ids in class_ids.values()
            ]
        )
        draws.append(draw)
    return draws


def aggregate_accuracy(rows: pd.DataFrame) -> dict[str, float]:
    """Reproduce the three overall accuracy definitions used in Chapter 5."""

    return {
        "mae": float(rows["absolute_error_sum"].sum() / rows["forecast_days"].sum()),
        "rmsse": float(rows["rmsse"].mean()),
        "wape": float(100.0 * rows["absolute_error_sum"].sum() / rows["actual_total"].sum()),
    }


def prepare_accuracy_by_series(metrics: pd.DataFrame) -> pd.DataFrame:
    """Aggregate origins within each series while retaining metric components."""

    return (
        metrics.groupby(["purpose", "model", "id"], as_index=False)
        .agg(
            forecast_days=("forecast_days", "sum"),
            actual_total=("actual_total", "sum"),
            absolute_error_sum=("absolute_error_sum", "sum"),
            rmsse=("rmsse", "mean"),
        )
    )


def point_accuracy_bootstrap(
    core_dir: Path,
    sample: pd.DataFrame,
    output_dir: Path,
) -> pd.DataFrame:
    """Paired series bootstrap for all point-accuracy model pairs."""

    metrics = pd.read_csv(core_dir / "point_forecast_series_origin_metrics.csv")
    metrics["model"] = normalise_model_names(metrics["model"])
    metrics["id"] = metrics["id"].astype(str)
    by_series = prepare_accuracy_by_series(metrics)

    rng = np.random.default_rng(RANDOM_SEED)
    draws = stratified_bootstrap_ids(sample, rng)
    result_rows: list[dict[str, object]] = []

    for purpose in ["validation", "final_test"]:
        purpose_rows = by_series.loc[by_series["purpose"].eq(purpose)].copy()
        model_tables = {
            model: purpose_rows.loc[purpose_rows["model"].eq(model)].set_index("id")
            for model in MODEL_ORDER
        }

        for model_a, model_b in itertools.combinations(MODEL_ORDER, 2):
            point_a = aggregate_accuracy(model_tables[model_a])
            point_b = aggregate_accuracy(model_tables[model_b])
            bootstrap_differences = {
                metric: np.empty(BOOTSTRAP_RESAMPLES, dtype=float)
                for metric in ["mae", "rmsse", "wape"]
            }

            for draw_number, selected_ids in enumerate(draws):
                draw_a = model_tables[model_a].loc[selected_ids]
                draw_b = model_tables[model_b].loc[selected_ids]
                aggregate_a = aggregate_accuracy(draw_a)
                aggregate_b = aggregate_accuracy(draw_b)
                for metric in bootstrap_differences:
                    bootstrap_differences[metric][draw_number] = (
                        aggregate_a[metric] - aggregate_b[metric]
                    )

            for metric, differences in bootstrap_differences.items():
                lower, upper = percentile_interval(differences)
                result_rows.append(
                    {
                        "purpose": purpose,
                        "model_a": model_a,
                        "model_b": model_b,
                        "metric": metric,
                        "point_difference_a_minus_b": point_a[metric] - point_b[metric],
                        "ci_lower": lower,
                        "ci_upper": upper,
                        "interval_excludes_zero": bool(lower > 0 or upper < 0),
                        "bootstrap_unit": "SKU-store series",
                        "bootstrap_resamples": BOOTSTRAP_RESAMPLES,
                        "random_seed": RANDOM_SEED,
                    }
                )

    results = pd.DataFrame(result_rows)
    results.to_csv(output_dir / "point_accuracy_paired_bootstrap.csv", index=False)
    return results


In [ ]:
point_bootstrap = point_accuracy_bootstrap(CORE_DIR, sample, OUTPUT_DIR)
print("Point-accuracy comparison rows:", len(point_bootstrap))
display(point_bootstrap.head(10))


## 5. ARIMA/SARIMA non-convergence sensitivity

This section documents the precise convergence counts and recomputes accuracy after either excluding affected series-origin observations or replacing their forecasts with the seasonal-naïve fallback.


In [ ]:
def arima_nonconvergence_sensitivity(
    core_dir: Path,
    sample: pd.DataFrame,
    output_dir: Path,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Summarise non-convergence and recompute accuracy under two treatments."""

    original = pd.read_csv(core_dir / "arima_sarima_fit_diagnostics.csv")
    additional = pd.read_csv(
        core_dir / "inventory_weekly_arima_sarima_additional_diagnostics.csv"
    )
    diagnostics = pd.concat(
        [original.assign(source="original"), additional.assign(source="additional")],
        ignore_index=True,
    )
    diagnostics["id"] = diagnostics["id"].astype(str)
    diagnostics["converged"] = diagnostics["converged"].astype(bool)
    diagnostics["fallback_used"] = diagnostics["fallback_used"].astype(bool)

    nonconverged = diagnostics.loc[~diagnostics["converged"]].copy()
    summary = pd.DataFrame(
        [
            {
                "fits_total": len(diagnostics),
                "fits_converged": int(diagnostics["converged"].sum()),
                "fits_nonconverged": len(nonconverged),
                "nonconverged_share_percent": 100.0 * len(nonconverged) / len(diagnostics),
                "affected_series": nonconverged["id"].nunique(),
                "fallback_fits": int(diagnostics["fallback_used"].sum()),
                "nonconverged_with_finite_aicc": int(np.isfinite(nonconverged["aicc"]).sum()),
                "nonconverged_with_recorded_error": int(
                    nonconverged["error_message"].notna().sum()
                ),
            }
        ]
    )
    summary.to_csv(output_dir / "arima_nonconvergence_summary.csv", index=False)

    metrics = pd.read_csv(core_dir / "point_forecast_series_origin_metrics.csv")
    metrics["model"] = normalise_model_names(metrics["model"])
    metrics["id"] = metrics["id"].astype(str)

    evaluation_nonconverged = nonconverged.loc[
        nonconverged["forecast_origin"].isin([1829, 1857, 1885, 1913]),
        ["id", "forecast_origin"],
    ].drop_duplicates()
    evaluation_nonconverged["nonconverged"] = True

    arima = metrics.loc[metrics["model"].eq("arima_sarima")].copy()
    seasonal = metrics.loc[metrics["model"].eq("seasonal_naive")].copy()
    merge_keys = ["purpose", "round", "forecast_origin", "id", "demand_class"]

    arima_flagged = arima.merge(
        evaluation_nonconverged,
        on=["id", "forecast_origin"],
        how="left",
    )
    arima_flagged["nonconverged"] = (
        arima_flagged["nonconverged"].eq(True)
    )

    seasonal_replacement = seasonal[merge_keys + [
        "forecast_days", "actual_total", "absolute_error_sum", "rmsse"
    ]].rename(
        columns={
            "forecast_days": "fallback_forecast_days",
            "actual_total": "fallback_actual_total",
            "absolute_error_sum": "fallback_absolute_error_sum",
            "rmsse": "fallback_rmsse",
        }
    )
    replacement = arima_flagged.merge(
        seasonal_replacement,
        on=merge_keys,
        how="left",
        validate="one_to_one",
    )

    treatment_rows: list[dict[str, object]] = []
    for purpose in ["validation", "final_test"]:
        retained = arima_flagged.loc[arima_flagged["purpose"].eq(purpose)].copy()
        excluded = retained.loc[~retained["nonconverged"]].copy()
        replaced = replacement.loc[replacement["purpose"].eq(purpose)].copy()
        mask = replaced["nonconverged"]
        for target, fallback in [
            ("forecast_days", "fallback_forecast_days"),
            ("actual_total", "fallback_actual_total"),
            ("absolute_error_sum", "fallback_absolute_error_sum"),
            ("rmsse", "fallback_rmsse"),
        ]:
            replaced.loc[mask, target] = replaced.loc[mask, fallback]

        for treatment, rows in [
            ("retained_finite_forecasts", retained),
            ("excluded_nonconverged_series_origins", excluded),
            ("seasonal_naive_fallback_replacement", replaced),
        ]:
            values = aggregate_accuracy(rows)
            treatment_rows.append(
                {
                    "purpose": purpose,
                    "treatment": treatment,
                    "series_origin_rows": len(rows),
                    "nonconverged_rows_affected": int(retained["nonconverged"].sum()),
                    **values,
                }
            )

    sensitivity = pd.DataFrame(treatment_rows)
    sensitivity.to_csv(output_dir / "arima_nonconvergence_accuracy_sensitivity.csv", index=False)

    affected_ids = set(nonconverged["id"])
    costs = pd.read_csv(core_dir / "inventory_series_cost_scenarios.csv")
    costs["model"] = normalise_model_names(costs["model"])
    costs["id"] = costs["id"].astype(str)
    baseline_costs = costs.loc[costs["scenario"].eq("baseline")].copy()
    fair_ids = set(sample["id"]) - affected_ids

    combined = baseline_costs.copy()
    combined = combined.loc[
        combined["id"].isin(fair_ids) & combined["quantile_level"].eq(0.99)
    ]
    inventory_sensitivity = (
        combined.groupby("model", as_index=False)
        .agg(
            total_cost=("total_cost", "sum"),
            total_demand=("total_demand", "sum"),
            fulfilled_units=("fulfilled_units", "sum"),
            series=("id", "nunique"),
        )
    )
    inventory_sensitivity["fill_rate"] = (
        inventory_sensitivity["fulfilled_units"] / inventory_sensitivity["total_demand"]
    )
    inventory_sensitivity.to_csv(
        output_dir / "arima_affected_series_inventory_sensitivity.csv", index=False
    )
    return summary, sensitivity


In [ ]:
arima_summary, arima_sensitivity = arima_nonconvergence_sensitivity(
    CORE_DIR,
    sample,
    OUTPUT_DIR,
)
display(arima_summary)
display(arima_sensitivity)


## 6. TiRex-2 calibration and quantile crossings

The analysis reports bootstrap uncertainty for empirical coverage, recognises the coverage interval induced by discrete demand and zeros, summarises crossing magnitudes, and checks whether monotone rearrangement materially changes pinball loss or coverage.


In [ ]:
def bootstrap_mean_by_series(
    values: pd.DataFrame,
    sample: pd.DataFrame,
    value_column: str,
) -> tuple[float, float]:
    """Stratified series-bootstrap interval for a per-series mean."""

    rng = np.random.default_rng(RANDOM_SEED)
    draws = stratified_bootstrap_ids(sample, rng)
    indexed = values.set_index("id")[value_column]
    estimates = np.array([indexed.loc[draw].mean() for draw in draws])
    return percentile_interval(estimates)


def tirex_calibration_and_crossings(
    quantile_dir: Path,
    sample: pd.DataFrame,
    output_dir: Path,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Calibration uncertainty, discrete-demand bands, and rearrangement check."""

    daily = pd.read_csv(quantile_dir / "tirex2_daily_quantile_evaluation_rows.csv")
    daily["id"] = daily["id"].astype(str)

    coverage_rows: list[dict[str, object]] = []
    for (purpose, quantile_level), group in daily.groupby(["purpose", "quantile_level"]):
        per_series = (
            group.assign(
                weak_covered=group["actual"].le(group["quantile_forecast"]),
                strict_covered=group["actual"].lt(group["quantile_forecast"]),
            )
            .groupby("id", as_index=False)
            .agg(
                weak_coverage=("weak_covered", "mean"),
                strict_coverage=("strict_covered", "mean"),
                zero_share=("actual", lambda x: x.eq(0).mean()),
            )
        )
        weak_lower, weak_upper = bootstrap_mean_by_series(
            per_series, sample, "weak_coverage"
        )
        coverage_rows.append(
            {
                "purpose": purpose,
                "quantile_level": quantile_level,
                "strict_coverage_p_y_less_than_q": per_series["strict_coverage"].mean(),
                "weak_coverage_p_y_less_or_equal_q": per_series["weak_coverage"].mean(),
                "weak_coverage_ci_lower": weak_lower,
                "weak_coverage_ci_upper": weak_upper,
                "nominal_within_discrete_quantile_band": bool(
                    per_series["strict_coverage"].mean()
                    <= quantile_level
                    <= per_series["weak_coverage"].mean()
                ),
                "actual_zero_share": per_series["zero_share"].mean(),
                "bootstrap_resamples": BOOTSTRAP_RESAMPLES,
                "random_seed": RANDOM_SEED,
            }
        )

    daily_coverage = pd.DataFrame(coverage_rows)
    daily_coverage.to_csv(output_dir / "tirex_daily_coverage_intervals.csv", index=False)

    protection = pd.read_csv(quantile_dir / "residual_based_protection_quantiles.csv")
    protection["id"] = protection["id"].astype(str)
    protection_rows: list[dict[str, object]] = []
    for (purpose, model, quantile_level), group in protection.groupby(
        ["purpose", "model", "quantile_level"]
    ):
        per_series = group.groupby("id", as_index=False).agg(coverage=("covered", "mean"))
        lower, upper = bootstrap_mean_by_series(per_series, sample, "coverage")
        protection_rows.append(
            {
                "purpose": purpose,
                "model": model,
                "quantile_level": quantile_level,
                "empirical_coverage": per_series["coverage"].mean(),
                "ci_lower": lower,
                "ci_upper": upper,
                "bootstrap_resamples": BOOTSTRAP_RESAMPLES,
                "random_seed": RANDOM_SEED,
            }
        )
    protection_coverage = pd.DataFrame(protection_rows)
    protection_coverage.to_csv(
        output_dir / "protection_coverage_intervals.csv", index=False
    )

    key_columns = [
        "id",
        "demand_class",
        "purpose",
        "round",
        "forecast_origin",
        "horizon",
        "target_day",
        "actual",
    ]
    wide = daily.pivot(index=key_columns, columns="quantile_level", values="quantile_forecast")
    wide = wide.sort_index(axis=1)
    original_values = wide.to_numpy(dtype=float)
    rearranged_values = np.sort(original_values, axis=1)
    adjacent_magnitudes = np.maximum(original_values[:, :-1] - original_values[:, 1:], 0.0)
    positive_magnitudes = adjacent_magnitudes[adjacent_magnitudes > 0]
    changed = np.abs(rearranged_values - original_values)

    long_rearranged = wide.copy()
    long_rearranged.iloc[:, :] = rearranged_values
    long_rearranged = (
        long_rearranged.stack(future_stack=True)
        .rename("rearranged_forecast")
        .reset_index()
    )
    comparison = daily.merge(
        long_rearranged,
        on=key_columns + ["quantile_level"],
        how="left",
        validate="one_to_one",
    )
    error = comparison["actual"] - comparison["rearranged_forecast"]
    alpha = comparison["quantile_level"]
    comparison["rearranged_pinball_loss"] = np.maximum(alpha * error, (alpha - 1) * error)
    comparison["rearranged_covered"] = comparison["actual"].le(
        comparison["rearranged_forecast"]
    )

    rearrangement_by_quantile = (
        comparison.groupby(["purpose", "quantile_level"], as_index=False)
        .agg(
            original_pinball_loss=("pinball_loss", "mean"),
            rearranged_pinball_loss=("rearranged_pinball_loss", "mean"),
            original_coverage=("covered", "mean"),
            rearranged_coverage=("rearranged_covered", "mean"),
        )
    )
    rearrangement_by_quantile["pinball_loss_change"] = (
        rearrangement_by_quantile["rearranged_pinball_loss"]
        - rearrangement_by_quantile["original_pinball_loss"]
    )
    rearrangement_by_quantile["coverage_change"] = (
        rearrangement_by_quantile["rearranged_coverage"]
        - rearrangement_by_quantile["original_coverage"]
    )
    rearrangement_by_quantile.to_csv(
        output_dir / "tirex_monotone_rearrangement_by_quantile.csv", index=False
    )

    crossing_summary = pd.DataFrame(
        [
            {
                "series_days": len(wide),
                "adjacent_quantile_comparisons": adjacent_magnitudes.size,
                "crossing_count": len(positive_magnitudes),
                "crossing_rate_percent": 100.0
                * len(positive_magnitudes)
                / adjacent_magnitudes.size,
                "affected_series_days": int((adjacent_magnitudes > 0).any(axis=1).sum()),
                "affected_series_day_share_percent": 100.0
                * (adjacent_magnitudes > 0).any(axis=1).mean(),
                "mean_positive_crossing_magnitude": positive_magnitudes.mean(),
                "median_positive_crossing_magnitude": np.median(positive_magnitudes),
                "p90_positive_crossing_magnitude": np.quantile(positive_magnitudes, 0.90),
                "p95_positive_crossing_magnitude": np.quantile(positive_magnitudes, 0.95),
                "p99_positive_crossing_magnitude": np.quantile(positive_magnitudes, 0.99),
                "maximum_positive_crossing_magnitude": positive_magnitudes.max(),
                "quantile_cells_changed_by_rearrangement": int((changed > 0).sum()),
                "mean_absolute_change_among_changed_cells": changed[changed > 0].mean(),
                "maximum_absolute_cell_change": changed.max(),
                "maximum_absolute_pinball_loss_change": rearrangement_by_quantile[
                    "pinball_loss_change"
                ].abs().max(),
                "maximum_absolute_coverage_change": rearrangement_by_quantile[
                    "coverage_change"
                ].abs().max(),
            }
        ]
    )
    crossing_summary.to_csv(output_dir / "tirex_crossing_rearrangement_summary.csv", index=False)

    class_crossings: list[dict[str, object]] = []
    index_frame = wide.index.to_frame(index=False)
    for demand_class in CLASS_ORDER:
        mask = index_frame["demand_class"].eq(demand_class).to_numpy()
        magnitudes = adjacent_magnitudes[mask]
        positive = magnitudes[magnitudes > 0]
        class_crossings.append(
            {
                "demand_class": demand_class,
                "adjacent_comparisons": magnitudes.size,
                "crossing_count": len(positive),
                "crossing_rate_percent": 100.0 * len(positive) / magnitudes.size,
                "affected_series_day_share_percent": 100.0
                * (magnitudes > 0).any(axis=1).mean(),
                "mean_positive_crossing_magnitude": positive.mean() if len(positive) else 0.0,
                "maximum_positive_crossing_magnitude": positive.max() if len(positive) else 0.0,
            }
        )
    pd.DataFrame(class_crossings).to_csv(
        output_dir / "tirex_crossings_by_demand_class.csv", index=False
    )
    return daily_coverage, protection_coverage, crossing_summary


In [ ]:
daily_coverage, protection_coverage, crossing_summary = (
    tirex_calibration_and_crossings(
        QUANTILE_DIR,
        sample,
        OUTPUT_DIR,
    )
)
display(daily_coverage)
display(protection_coverage)
display(crossing_summary)


## 7. Terminal inventory state

Final on-hand inventory and outstanding pipeline inventory are reported separately for the baseline and principal review-period/lead-time comparisons.


In [ ]:
def terminal_inventory_summary(core_dir: Path, output_dir: Path) -> pd.DataFrame:
    """Report terminal on-hand and pipeline inventory for principal policies."""

    baseline = pd.read_csv(core_dir / "inventory_summary.csv")
    baseline["model"] = normalise_model_names(baseline["model"])
    baseline = baseline.loc[
        baseline["scenario"].eq("baseline")
        & baseline["demand_class"].eq("Overall")
        & baseline["quantile_level"].eq(0.99)
    ].copy()
    baseline["review_period"] = 7
    baseline["lead_time"] = 21
    baseline["mean_terminal_on_hand"] = baseline["ending_inventory"] / baseline["series"]
    baseline["mean_terminal_pipeline"] = baseline["ending_pipeline"] / baseline["series"]
    baseline["mean_terminal_inventory_position"] = (
        baseline["mean_terminal_on_hand"] + baseline["mean_terminal_pipeline"]
    )

    timing = pd.read_csv(core_dir / "inventory_policy_timing_series_results.csv")
    timing["model"] = normalise_model_names(timing["model"])
    principal = pd.DataFrame(
        [
            {"review_period": 14, "lead_time": 7, "model": "ets"},
            {"review_period": 14, "lead_time": 14, "model": "xgboost"},
            {"review_period": 21, "lead_time": 7, "model": "xgboost"},
        ]
    )
    timing = timing.loc[timing["quantile_level"].eq(0.99)].merge(
        principal,
        on=["review_period", "lead_time", "model"],
        how="inner",
    )
    timing_summary = (
        timing.groupby(["review_period", "lead_time", "model", "quantile_level"], as_index=False)
        .agg(
            total_cost=("total_cost", "sum"),
            demand=("demand_units", "sum"),
            fulfilled=("sales_units", "sum"),
            series=("id", "nunique"),
            mean_terminal_on_hand=("ending_inventory", "mean"),
            mean_terminal_pipeline=("ending_pipeline", "mean"),
        )
    )
    timing_summary["fill_rate"] = timing_summary["fulfilled"] / timing_summary["demand"]
    timing_summary["mean_terminal_inventory_position"] = (
        timing_summary["mean_terminal_on_hand"]
        + timing_summary["mean_terminal_pipeline"]
    )

    baseline_output = baseline[
        [
            "review_period",
            "lead_time",
            "model",
            "quantile_level",
            "total_cost",
            "fill_rate",
            "series",
            "mean_terminal_on_hand",
            "mean_terminal_pipeline",
            "mean_terminal_inventory_position",
        ]
    ]
    timing_output = timing_summary[baseline_output.columns]
    output = pd.concat([baseline_output, timing_output], ignore_index=True)
    output["protection_period"] = output["review_period"] + output["lead_time"]
    output = output[
        [
            "review_period",
            "lead_time",
            "protection_period",
            "model",
            "quantile_level",
            "total_cost",
            "fill_rate",
            "mean_terminal_on_hand",
            "mean_terminal_pipeline",
            "mean_terminal_inventory_position",
            "series",
        ]
    ].sort_values(["review_period", "lead_time", "total_cost"])
    output.to_csv(output_dir / "principal_terminal_inventory.csv", index=False)
    return output


In [ ]:
terminal = terminal_inventory_summary(CORE_DIR, OUTPUT_DIR)
display(terminal)


## 8. Cluster-bootstrap robustness

The original series-level intervals are complemented by resampling at product and store level. These checks retain dependence among observations sharing the selected cluster identifier.


In [ ]:
def cluster_bootstrap_difference(
    rows: pd.DataFrame,
    cluster_column: str,
    cost_a: str,
    cost_b: str,
    demand_column: str,
    sales_a: str,
    sales_b: str,
    rng: np.random.Generator,
) -> dict[str, float]:
    """Cluster bootstrap for a cost and fill-rate difference."""

    grouped = rows.groupby(cluster_column, as_index=False).agg(
        series=("id", "nunique"),
        cost_a=(cost_a, "sum"),
        cost_b=(cost_b, "sum"),
        demand=(demand_column, "sum"),
        sales_a=(sales_a, "sum"),
        sales_b=(sales_b, "sum"),
    )
    cluster_ids = grouped[cluster_column].to_numpy()
    lookup = grouped.set_index(cluster_column)
    target_series = rows["id"].nunique()

    cost_differences = np.empty(BOOTSTRAP_RESAMPLES)
    fill_differences = np.empty(BOOTSTRAP_RESAMPLES)
    for draw_number in range(BOOTSTRAP_RESAMPLES):
        chosen = rng.choice(cluster_ids, size=len(cluster_ids), replace=True)
        draw = lookup.loc[chosen]
        scale = target_series / draw["series"].sum()
        cost_differences[draw_number] = scale * (
            draw["cost_a"].sum() - draw["cost_b"].sum()
        )
        fill_differences[draw_number] = 100.0 * (
            draw["sales_a"].sum() / draw["demand"].sum()
            - draw["sales_b"].sum() / draw["demand"].sum()
        )

    cost_lower, cost_upper = percentile_interval(cost_differences)
    fill_lower, fill_upper = percentile_interval(fill_differences)
    return {
        "clusters": len(cluster_ids),
        "cost_ci_lower": cost_lower,
        "cost_ci_upper": cost_upper,
        "fill_difference_ci_lower_pp": fill_lower,
        "fill_difference_ci_upper_pp": fill_upper,
    }


def clustered_inventory_robustness(
    core_dir: Path,
    sample: pd.DataFrame,
    output_dir: Path,
) -> pd.DataFrame:
    """Product- and store-cluster checks for the two principal comparisons."""

    sample_map = sample[["id", "item_id", "store_id"]].copy()

    costs = pd.read_csv(core_dir / "inventory_series_cost_scenarios.csv")
    costs["model"] = normalise_model_names(costs["model"])
    costs["id"] = costs["id"].astype(str)
    baseline = costs.loc[
        costs["scenario"].eq("baseline") & costs["quantile_level"].eq(0.99)
    ].copy()

    a = baseline.loc[baseline["model"].eq("seasonal_naive")][
        ["id", "total_cost", "total_demand", "fulfilled_units"]
    ].rename(
        columns={
            "total_cost": "cost_a",
            "total_demand": "demand",
            "fulfilled_units": "sales_a",
        }
    )
    b = baseline.loc[baseline["model"].eq("xgboost")][
        ["id", "total_cost", "fulfilled_units"]
    ].rename(columns={"total_cost": "cost_b", "fulfilled_units": "sales_b"})
    baseline_pair = a.merge(b, on="id", validate="one_to_one").merge(
        sample_map, on="id", validate="one_to_one"
    )

    timing = pd.read_csv(core_dir / "inventory_policy_timing_series_results.csv")
    timing["model"] = normalise_model_names(timing["model"])
    timing["id"] = timing["id"].astype(str)
    a_timing = timing.loc[
        timing["quantile_level"].eq(0.99)
        & timing["model"].eq("ets")
        & timing["review_period"].eq(14)
        & timing["lead_time"].eq(7)
    ][["id", "total_cost", "demand_units", "sales_units"]].rename(
        columns={
            "total_cost": "cost_a",
            "demand_units": "demand",
            "sales_units": "sales_a",
        }
    )
    b_timing = timing.loc[
        timing["quantile_level"].eq(0.99)
        & timing["model"].eq("xgboost")
        & timing["review_period"].eq(14)
        & timing["lead_time"].eq(14)
    ][["id", "total_cost", "sales_units"]].rename(
        columns={"total_cost": "cost_b", "sales_units": "sales_b"}
    )
    timing_pair = a_timing.merge(b_timing, on="id", validate="one_to_one").merge(
        sample_map, on="id", validate="one_to_one"
    )

    result_rows: list[dict[str, object]] = []
    for comparison, rows, label_a, label_b in [
        (
            "baseline_q099",
            baseline_pair,
            "seasonal_naive_R7_L21",
            "xgboost_R7_L21",
        ),
        (
            "best_timing_q099",
            timing_pair,
            "ets_R14_L7",
            "xgboost_R14_L14",
        ),
    ]:
        point_cost = rows["cost_a"].sum() - rows["cost_b"].sum()
        point_fill = 100.0 * (
            rows["sales_a"].sum() / rows["demand"].sum()
            - rows["sales_b"].sum() / rows["demand"].sum()
        )
        for cluster_column in ["item_id", "store_id"]:
            rng = np.random.default_rng(RANDOM_SEED)
            interval = cluster_bootstrap_difference(
                rows=rows,
                cluster_column=cluster_column,
                cost_a="cost_a",
                cost_b="cost_b",
                demand_column="demand",
                sales_a="sales_a",
                sales_b="sales_b",
                rng=rng,
            )
            result_rows.append(
                {
                    "comparison": comparison,
                    "configuration_a": label_a,
                    "configuration_b": label_b,
                    "cluster_unit": cluster_column,
                    "point_cost_difference_a_minus_b": point_cost,
                    "point_fill_difference_a_minus_b_pp": point_fill,
                    **interval,
                    "bootstrap_resamples": BOOTSTRAP_RESAMPLES,
                    "random_seed": RANDOM_SEED,
                }
            )

    results = pd.DataFrame(result_rows)
    results.to_csv(output_dir / "inventory_cluster_bootstrap_robustness.csv", index=False)
    return results


In [ ]:
clustered = clustered_inventory_robustness(CORE_DIR, sample, OUTPUT_DIR)
display(clustered)


## 9. Cost-fill-rate and Pareto-frontier figure

All 15 baseline model-quantile combinations are plotted in cost-fill-rate space. A point is Pareto-efficient when no alternative has both a lower or equal cost and a higher or equal fill rate, with at least one strict improvement.


In [ ]:
def pareto_frontier(data: pd.DataFrame) -> pd.Series:
    """Return a Boolean mask for lower-cost/higher-fill Pareto efficiency."""

    efficient = np.ones(len(data), dtype=bool)
    cost = data["total_cost"].to_numpy()
    fill = data["fill_rate"].to_numpy()
    for index in range(len(data)):
        dominated = (
            (cost <= cost[index])
            & (fill >= fill[index])
            & ((cost < cost[index]) | (fill > fill[index]))
        )
        if dominated.any():
            efficient[index] = False
    return pd.Series(efficient, index=data.index)


def cost_fill_pareto_figure(core_dir: Path, output_dir: Path, figure_dir: Path) -> pd.DataFrame:
    """Create the baseline cost--fill-rate plot promised in the methodology."""

    summary = pd.read_csv(core_dir / "inventory_summary.csv")
    summary["model"] = normalise_model_names(summary["model"])
    data = summary.loc[
        summary["scenario"].eq("baseline") & summary["demand_class"].eq("Overall")
    ].copy()
    data["pareto_recomputed"] = pareto_frontier(data)
    data.to_csv(output_dir / "baseline_cost_fill_pareto_points.csv", index=False)

    colours = {
        "seasonal_naive": "#0072B2",
        "ets": "#009E73",
        "arima_sarima": "#CC79A7",
        "xgboost": "#D55E00",
        "tirex2": "#6A3D9A",
    }
    markers = {0.90: "o", 0.95: "s", 0.99: "D"}

    fig, ax = plt.subplots(figsize=(8.3, 5.4), constrained_layout=True)
    for model in MODEL_ORDER:
        rows = data.loc[data["model"].eq(model)].sort_values("quantile_level")
        ax.plot(
            100.0 * rows["fill_rate"],
            rows["total_cost"],
            color=colours[model],
            linewidth=1.3,
            alpha=0.7,
            zorder=1,
        )
        for _, row in rows.iterrows():
            ax.scatter(
                100.0 * row["fill_rate"],
                row["total_cost"],
                color=colours[model],
                marker=markers[float(row["quantile_level"])],
                s=66,
                edgecolor="white",
                linewidth=0.7,
                zorder=3,
            )

    frontier = data.loc[data["pareto_recomputed"]].sort_values("fill_rate")
    ax.plot(
        100.0 * frontier["fill_rate"],
        frontier["total_cost"],
        color="#111111",
        linewidth=2.2,
        linestyle="--",
        label="Pareto frontier",
        zorder=2,
    )

    annotation_offsets = {
        "seasonal_naive": (-108, 17),
        "xgboost": (-22, 24),
    }
    for _, row in frontier.iterrows():
        label = f"{MODEL_LABELS[row['model']]} (0.99)"
        ax.annotate(
            label,
            xy=(100.0 * row["fill_rate"], row["total_cost"]),
            xytext=annotation_offsets[row["model"]],
            textcoords="offset points",
            fontsize=9,
            fontweight="bold",
            arrowprops={"arrowstyle": "-", "color": "#333333", "lw": 0.8},
        )

    model_handles = [
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="none",
            markerfacecolor=colours[model],
            markeredgecolor="white",
            markersize=7,
            label=MODEL_LABELS[model],
        )
        for model in MODEL_ORDER
    ]
    quantile_handles = [
        plt.Line2D(
            [0],
            [0],
            marker=markers[level],
            color="#555555",
            linestyle="none",
            markerfacecolor="white",
            markersize=7,
            label=f"Quantile {level:.2f}",
        )
        for level in [0.90, 0.95, 0.99]
    ]
    frontier_handle = plt.Line2D(
        [0], [0], color="#111111", linestyle="--", linewidth=2.2, label="Pareto frontier"
    )
    ax.legend(
        handles=model_handles + quantile_handles + [frontier_handle],
        loc="upper right",
        fontsize=8,
        frameon=True,
        ncol=2,
    )
    ax.set_xlabel("Realised fill rate (%)")
    ax.set_ylabel("Total comparative simulation cost")
    ax.set_title("Baseline cost--service trade-off across model--quantile combinations")
    ax.grid(True, color="#D9D9D9", linewidth=0.6, alpha=0.8)
    ax.set_axisbelow(True)
    ax.margins(x=0.08, y=0.10)

    figure_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(figure_dir / "chapter5_baseline_cost_fill_pareto.png", dpi=300)
    fig.savefig(figure_dir / "chapter5_baseline_cost_fill_pareto.pdf")
    plt.close(fig)
    return data


In [ ]:
pareto_points = cost_fill_pareto_figure(
    CORE_DIR,
    OUTPUT_DIR,
    FIGURE_DIR,
)
display(
    pareto_points.loc[pareto_points["pareto_recomputed"]]
    [["model", "quantile_level", "total_cost", "fill_rate"]]
)


## 10. Final implementation checks

The final cell verifies the expected dimensions and key identities of every revision output. All checks must equal `True`.


In [ ]:
def write_checks(
    output_dir: Path,
    point_bootstrap: pd.DataFrame,
    arima_summary: pd.DataFrame,
    daily_coverage: pd.DataFrame,
    protection_coverage: pd.DataFrame,
    crossing_summary: pd.DataFrame,
    terminal: pd.DataFrame,
    clustered: pd.DataFrame,
    pareto_points: pd.DataFrame,
) -> pd.DataFrame:
    """Write a compact validation log for the revision analyses."""

    checks = pd.DataFrame(
        {
            "check": [
                "Point-accuracy bootstrap has all pair-purpose-metric rows",
                "ARIMA diagnostics cover 11,000 fits",
                "TiRex daily coverage contains both purposes and nine quantiles",
                "Protection coverage contains five models and three quantiles",
                "Quantile rearrangement removes crossings without material score change",
                "Terminal table includes baseline and principal timing configurations",
                "Cluster robustness contains product and store resampling",
                "Pareto plot contains all fifteen baseline points",
                "Exactly two baseline Pareto-efficient points",
            ],
            "passed": [
                len(point_bootstrap) == 2 * 10 * 3,
                int(arima_summary.loc[0, "fits_total"]) == 11_000,
                len(daily_coverage) == 2 * 9,
                len(protection_coverage) == 2 * 5 * 3,
                float(crossing_summary.loc[0, "maximum_absolute_pinball_loss_change"])
                < 0.0001,
                len(terminal) == 8,
                set(clustered["cluster_unit"]) == {"item_id", "store_id"},
                len(pareto_points) == 15,
                int(pareto_points["pareto_recomputed"].sum()) == 2,
            ],
        }
    )
    checks.to_csv(output_dir / "revision_analysis_checks.csv", index=False)
    if not checks["passed"].all():
        failed = checks.loc[~checks["passed"], "check"].tolist()
        raise AssertionError(f"Revision analysis checks failed: {failed}")
    return checks


In [ ]:
checks = write_checks(
    OUTPUT_DIR,
    point_bootstrap,
    arima_summary,
    daily_coverage,
    protection_coverage,
    crossing_summary,
    terminal,
    clustered,
    pareto_points,
)

print("Revision analyses completed successfully.")
print("Outputs:", OUTPUT_DIR)
print("Figures:", FIGURE_DIR)
display(checks)
